# 소비습관 (Habit) 모델

# 소비 패턴(Habit) 변화 분석을 통한 라이프스타일 전이 예측
> **목적**: 온라인/오프라인 비중 및 업종별 결제 패턴 변화를 분석하여 고객의 라이프스타일 변화로 인한 카드 교체 가능성 예측

### 주요 분석 스토리라인:
1. **가설 설정**: 특정 업종(유틸리티 등)만 남고 쇼핑/교육 비중이 줄어드는 현상은 주결제 수단 변경의 전조 증상이라 가정
2. **피처 엔지니어링**: 온라인 비중 변화량, 업종별 소비 비중, Ticket Size 변화 등 패턴 중심 변수 **산출**
3. **알고리즘 최적화**: 
   - 범주형 변수 및 비중 데이터 처리에 특화된 **CatBoost**를 최종 엔진으로 **선정**
   - 앙상블 조합(Stacking)을 통한 패턴 분류 성능의 극대화 **실현**
4. **인사이트 도출**: SHAP 분석을 통해 이탈에 기여하는 핵심 소비 업종 및 채널 **식별**
5. **데이터 연동**: 패턴 변화 위험군 식별을 위한 최종 예측 스코어 및 프로파일 **저장**

In [ ]:
# STEP 0. 라이브러리 및 한글 폰트 설정 

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import gc
import polars as pl
from scipy import stats
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import TimeSeriesSplit
from sklearn.inspection import permutation_importance
from sklearn.metrics import (
    classification_report, f1_score, precision_recall_curve, 
    auc, recall_score, precision_score, average_precision_score,
    calibration_curve, confusion_matrix
)
import shap

# 한글 폰트 설정 (Windows: Malgun Gothic)
plt.rc('font', family='Malgun Gothic')
plt.rcParams['axes.unicode_minus'] = False

print("STEP 0: 환경 설정 및 라이브러리 로드 완료")

In [ ]:
# STEP 1. 환경 설정 및 데이터 로드

# 통일된 마스터 데이터셋 로드
df = pl.read_parquet('final_master_dataset.parquet').to_pandas()

print(f"STEP 1: 데이터 로드 완료. 데이터 규모: {df.shape}")
# 이탈률 분포 확인 (클래스 불균형 체크)
print(f"이탈률 분포 확인: {df['is_churn'].value_counts(normalize=True)[1]:.2%}")

In [ ]:
# STEP 2. Y값 정의 및 변수 및 파생변수 생성

def engineer_habit_features(df):
    print("STEP 2: 소비 패턴 중심 피처 엔지니어링 수행 중...")
    eps = 1e-9
    
    # 1. 소비 채널 및 업종 비중 (Habit 핵심)
    df['온라인_비중'] = df['이용금액_온라인_B0M'] / (df['이용금액_신용_B0M'] + eps)
    df['쇼핑_비중'] = df['이용금액_쇼핑_B0M'] / (df['이용금액_신용_B0M'] + eps)
    df['유틸리티_비중'] = df['이용금액_유틸리티_B0M'] / (df['이용금액_신용_B0M'] + eps)
    
    # 2. 소비 성향 및 단가 지표
    df['Ticket_Size'] = df['이용금액_신용_B0M'] / (df['이용건수_신용_B0M'] + eps)
    df['건당_온라인금액'] = df['이용금액_온라인_B0M'] / (df['이용건수_온라인_B0M'] + eps)
    
    # 3. 패턴 변화 추세 지표 (전달 대비 온라인 비중 변화 등)
    prev_online_ratio = df['이용금액_온라인_B1M'] / (df['이용금액_신용_B1M'] + eps)
    df['온라인비중_변화량'] = df['온라인_비중'] - prev_online_ratio
    
    # 4. Y값 라벨링 재확인 (2019.01 완전 이탈 여부)
    # df['is_churn'] = (df['이용금액_201901'] <= 0).astype(int)
    
    return df

df = engineer_habit_features(df)
df = df.replace([np.inf, -np.inf], 0).fillna(0)

In [ ]:
# STEP 3. 피처 목록 확정

habit_features = [
    '온라인_비중', '쇼핑_비중', '유틸리티_비중', 'Ticket_Size', 
    '이용금액_신용_B0M', '건당_온라인금액', '온라인비중_변화량', 
    '연체여부'
]

X = df[habit_features]
y = df['is_churn']

print(f"STEP 3: 소비습관 핵심 변수 {len(habit_features)}개 확정")

In [ ]:
# STEP 4. 베이스모델(Random Forest) 및 모델링 (타임스플릿 적용) 

print("STEP 4: RF 베이스라인 학습 및 시계열 검증 진행...")

tscv = TimeSeriesSplit(n_splits=3)
rf_base = RandomForestClassifier(
    n_estimators=200, 
    max_depth=10, 
    class_weight='balanced', 
    random_state=42,
    n_jobs=-1
)

# 시계열 교차 검증 실행
for i, (train_idx, val_idx) in enumerate(tscv.split(X)):
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
    rf_base.fit(X_train, y_train)
    print(f"  - Fold {i+1} 학습 완료")

In [ ]:
# STEP 5. 피처 목록 확정 (SHAP, Permutation Importance, TopK Lift)

print("STEP 5: SHAP 및 비즈니스 리프트(Lift) 분석...")

# 1. SHAP Value 시각화
explainer = shap.TreeExplainer(rf_base)
shap_values = explainer.shap_values(X.sample(1000, random_state=42))
shap.summary_plot(shap_values, X.sample(1000, random_state=42))

# 2. Permutation Importance
perm_imp = permutation_importance(rf_base, X.sample(500), y.sample(500), n_repeats=5, random_state=42)
sorted_idx = perm_imp.importances_mean.argsort()
plt.figure(figsize=(10, 6))
plt.barh(X.columns[sorted_idx], perm_imp.importances_mean[sorted_idx])
plt.title("Permutation Importance (Habit)")
plt.show()

# 3. Top-K Lift 산출 (상위 10% 기준)
all_probs = rf_base.predict_proba(X)[:, 1]
lift_df = pd.DataFrame({'prob': all_probs, 'actual': y}).sort_values('prob', ascending=False)
top_10_actual = lift_df.head(int(len(lift_df)*0.1))['actual'].mean()
print(f"Habit Model Top 10% Lift: {top_10_actual / y.mean():.2f}배")

In [ ]:
# STEP 6. 모델 고도화를 위한 스태킹 진행 (RF, LightGBM, Catboost, XGboost 4개 사용)

print("STEP 6: 4개 엔진 기반 스태킹 앙상블 수행...")

stacking_models = {
    'RF': rf_base,
    'XGB': XGBClassifier(n_estimators=200, scale_pos_weight=5, random_state=42),
    'LGBM': LGBMClassifier(n_estimators=200, scale_pos_weight=5, random_state=42),
    'CAT': CatBoostClassifier(iterations=200, auto_class_weights='Balanced', verbose=0, random_state=42)
}

oof_preds = pd.DataFrame(index=df.index)

for name, model in stacking_models.items():
    print(f"  - {name} 엔진 학습 중...")
    model.fit(X, y)
    oof_preds[name] = model.predict_proba(X)[:, 1]

# Meta-Learner: ElasticNet (Logistic Regression)
meta_learner = LogisticRegression(penalty='elasticnet', solver='saga', l1_ratio=0.5)
meta_learner.fit(oof_preds, y)
final_stacking_probs = meta_learner.predict_proba(oof_preds)[:, 1]

In [ ]:
# STEP 7. 주제별 엔진 선정 (소비습관 - Catboost 선정)

print("STEP 7: 소비습관 최종 엔진(CatBoost) 모델링...")

final_engine = CatBoostClassifier(
    iterations=500,
    learning_rate=0.05,
    depth=6,
    auto_class_weights='Balanced',
    verbose=0,
    random_state=42
)

final_engine.fit(X, y)
final_habit_probs = final_engine.predict_proba(X)[:, 1]

In [ ]:
# STEP 8. 주제별 선정된 엔진 기반 모델링 후 이탈자 저장

print("STEP 8: 마케팅 통합을 위한 예측 결과 저장 (Habit)...")

result_df = df[['발급회원번호', 'is_churn']].copy()
result_df['habit_prob'] = final_habit_probs

# 결과 저장
result_df.to_csv('habit_churn_full_results.csv', index=False, encoding='utf-8-sig')
print("habit_churn_full_results.csv 저장 완료.")

In [ ]:
# STEP 9. 모델 검증 (PSI, Calibration Curve)

print("STEP 9: 모델 안정성 및 신뢰도 진단...")

# 1. Calibration Curve (신뢰도 검정)
prob_true, prob_pred = calibration_curve(y, final_habit_probs, n_bins=10)
plt.figure(figsize=(8, 6))
plt.plot(prob_pred, prob_true, marker='s', label='Habit CatBoost')
plt.plot([0, 1], [0, 1], linestyle='--', color='gray')
plt.title('Habit Model Calibration (Reliability)')
plt.xlabel('Predicted Probability')
plt.ylabel('Actual Fraction of Positives')
plt.legend()
plt.show()

# 2. PSI (Population Stability Index) 산출 함수
def calculate_psi(expected, actual, buckets=10):
    e_counts = np.histogram(expected, buckets)[0] / len(expected)
    a_counts = np.histogram(actual, buckets)[0] / len(actual)
    def psi_calc(e, a):
        if a == 0: a = 0.0001
        if e == 0: e = 0.0001
        return (e - a) * np.log(e / a)
    return sum([psi_calc(e_counts[i], a_counts[i]) for i in range(buckets)])

psi_val = calculate_psi(y[:5000], y[5000:10000])
print(f"PSI Score: {psi_val:.4f}")

In [ ]:
# STEP 10. 모델 검증 PR Curve 분석

print("STEP 10: 최종 PR-AUC 성능 검증...")

precision, recall, _ = precision_recall_curve(y, final_habit_probs)
pr_auc = auc(recall, precision)
ap_score = average_precision_score(y, final_habit_probs)

plt.figure(figsize=(8, 6))
plt.plot(recall, precision, color='darkorange', lw=3, label=f'PR-AUC = {pr_auc:.4f}')
plt.axhline(y.mean(), color='blue', linestyle='--', label=f'Baseline ({y.mean():.2%})')
plt.title('Habit Model Precision-Recall Curve')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.legend()
plt.show()

print(f"Final AP Score: {ap_score:.4f}")